# SAME 四类数据集梳理

这个 notebook 面向当前仓库里的 SAME 复现实验，回答四个问题：

1. 四类数据集的原始文件各自存放在哪里。
2. SAME 在加载时对每类数据做了什么预处理。
3. 各类数据的指令长什么样，尤其是 CVDN 的对话角色如何区分。
4. 四类数据集的目标本质上有什么区别。

默认与当前项目约定一致，使用 `test-v1` 内核。

In [ ]:
from __future__ import annotations

import json
import statistics
from pathlib import Path

import jsonlines
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_ROOT = ROOT / 'data' / 'same'
CODE_ROOT = ROOT / 'third_party' / 'SAME' / 'src' / 'tasks' / 'datasets'

DATASET_INFO = {
    'R2R': {
        'path': DATA_ROOT / 'R2R' / 'R2R_val_unseen_enc.json',
        'loader': CODE_ROOT / 'r2r.py',
        'format': 'json',
        'task_nature': '纯导航，到达目标 viewpoint 附近',
        'goal_unit': 'viewpoint / path endpoint',
    },
    'REVERIE': {
        'path': DATA_ROOT / 'REVERIE' / 'REVERIE_val_unseen_enc.json',
        'loader': CODE_ROOT / 'reverie.py',
        'format': 'json',
        'task_nature': '导航到能看见目标物体的 viewpoint，并评估物体 ID',
        'goal_unit': 'object-visible viewpoint + object id',
    },
    'CVDN': {
        'path': DATA_ROOT / 'CVDN' / 'val_unseen.json',
        'loader': CODE_ROOT / 'cvdn.py',
        'format': 'json',
        'task_nature': '对话式导航，到达目标房间附近',
        'goal_unit': 'end pano / goal room',
    },
    'SOON': {
        'path': DATA_ROOT / 'SOON' / 'val_unseen_house_enc_pseudo_obj_ade30k_label.jsonl',
        'loader': CODE_ROOT / 'soon.py',
        'format': 'jsonl',
        'task_nature': '先导航到目标 viewpoint，再做目标方向/框定位',
        'goal_unit': 'goal viewpoint + object direction / bbox polygon',
    },
}

pd.DataFrame([
    {
        'dataset': name,
        'annotation_file': str(info['path'].relative_to(ROOT)),
        'loader': str(info['loader'].relative_to(ROOT)),
        'task_nature': info['task_nature'],
        'goal_unit': info['goal_unit'],
    }
    for name, info in DATASET_INFO.items()
])

In [ ]:
def load_first_record(dataset_name: str):
    info = DATASET_INFO[dataset_name]
    if info['format'] == 'json':
        return json.load(open(info['path']))[0]
    with jsonlines.open(info['path']) as reader:
        return next(reader.iter())


def reconstruct_cvdn_instruction(item: dict) -> str:
    prefix = f"The goal room contains a {item['target']}.\n"
    turns = []
    for turn in item['dialog_history']:
        msg = turn['message'] if turn['message'][-1] in '?.' else turn['message'] + '.'
        role_prefix = 'Question: ' if turn['role'] == 'navigator' else 'Answer: '
        turns.append(role_prefix + msg + '\n')
    text = prefix + ''.join(turns)
    return text[:-1] if text.endswith('\n') else text


def summarize_preprocessing(dataset_name: str, item: dict) -> dict:
    if dataset_name == 'R2R':
        return {
            'sample expansion': '一个 path 样本里的 3 条 instructions 会拆成 3 个训练样本',
            'selected text field': "item['instructions'][j]",
            'selected encoding': "item['instr_encodings'][j][:200]",
            'important note': '代码默认切到 200 token，而不是 config 里的 80',
        }
    if dataset_name == 'REVERIE':
        return {
            'sample expansion': '和 R2R 一样，按 instructions 拆样本',
            'selected text field': "item['instructions'][j]",
            'selected encoding': "item['instr_encodings'][j]",
            'important note': '训练时如果开启 multi_endpoints，会随机采样一个 end_vps 并改写 path',
        }
    if dataset_name == 'CVDN':
        return {
            'sample expansion': '不按多 instruction 拆分，一个对话 episode 就是一个样本',
            'selected text field': 'target + dialog_history 拼接后的单条 instruction',
            'selected encoding': '运行时用 bert-base-uncased tokenizer 现算',
            'important note': '在 __getitem__ 里如果 instruction 超过 128 个空格分词词元，会把 instruction 文本截短到 128 词，但不会同步重算 instr_encoding',
        }
    if dataset_name == 'SOON':
        return {
            'sample expansion': '按 instructions 拆样本，但每个 item 常常只有 1 条',
            'selected text field': "item['instructions'][j]['full']",
            'selected encoding': "item['instr_encodings'][j]['full'][:100]",
            'important note': '训练时会随机选一个 end_image_id，把 path 改写成到该 endpoint 的最短路',
        }
    raise KeyError(dataset_name)


sample_rows = []
for dataset_name in DATASET_INFO:
    item = load_first_record(dataset_name)
    row = {'dataset': dataset_name}
    row.update(summarize_preprocessing(dataset_name, item))
    sample_rows.append(row)

pd.DataFrame(sample_rows)

In [ ]:
samples = {}

r2r = load_first_record('R2R')
samples['R2R'] = {
    'path_id': r2r['path_id'],
    'scan': r2r['scan'],
    'path': r2r['path'],
    'instructions_count': len(r2r['instructions']),
    'instruction_example': r2r['instructions'][0],
    'encoding_len_example': len(r2r['instr_encodings'][0]),
}

reverie = load_first_record('REVERIE')
samples['REVERIE'] = {
    'path_id': reverie.get('path_id', reverie.get('id')),
    'scan': reverie['scan'],
    'path': reverie['path'],
    'objId': reverie.get('objId'),
    'instructions_count': len(reverie['instructions']),
    'instruction_example': reverie['instructions'][0],
    'encoding_len_example': len(reverie['instr_encodings'][0]),
}

cvdn = load_first_record('CVDN')
samples['CVDN'] = {
    'inst_idx': cvdn['inst_idx'],
    'scan': cvdn['scan'],
    'target': cvdn['target'],
    'start_pano': cvdn['start_pano'],
    'planner_path': cvdn.get('planner_path'),
    'dialog_turns': cvdn['dialog_history'][:6],
    'reconstructed_instruction': reconstruct_cvdn_instruction(cvdn),
}

soon = load_first_record('SOON')
samples['SOON'] = {
    'path_id': soon['path_id'],
    'scan': soon['scan'],
    'path': soon['path'],
    'instructions_count': len(soon['instructions']),
    'instruction_keys': list(soon['instructions'][0].keys()),
    'instruction_full': soon['instructions'][0]['full'],
    'encoding_len_example': len(soon['instr_encodings'][0]['full']),
    'bbox_example_keys': list(soon['bboxes'][0].keys()),
}

for name, payload in samples.items():
    display(Markdown(f'## {name}'))
    for key, value in payload.items():
        display(Markdown(f'**{key}**'))
        display(value)


In [ ]:
def collect_stats():
    rows = []

    r2r_data = json.load(open(DATASET_INFO['R2R']['path']))
    r2r_lens = [len(enc) for item in r2r_data for enc in item['instr_encodings']]
    rows.append({
        'dataset': 'R2R',
        'instances_after_loading': len(r2r_lens),
        'len_unit': 'tokenized ids',
        'min': min(r2r_lens),
        'median': statistics.median(r2r_lens),
        'max': max(r2r_lens),
        'truncation_rule': 'load_data 里切到 200',
        'truncated_count': sum(x > 200 for x in r2r_lens),
        'note': f">80 的有 {sum(x > 80 for x in r2r_lens)} 条，但代码实际没有切到 80",
    })

    reverie_data = json.load(open(DATASET_INFO['REVERIE']['path']))
    reverie_lens = [len(enc) for item in reverie_data for enc in item['instr_encodings']]
    rows.append({
        'dataset': 'REVERIE',
        'instances_after_loading': len(reverie_lens),
        'len_unit': 'tokenized ids',
        'min': min(reverie_lens),
        'median': statistics.median(reverie_lens),
        'max': max(reverie_lens),
        'truncation_rule': '不在 loader 里额外截断',
        'truncated_count': 0,
        'note': 'val_unseen 样本里最长 60',
    })

    cvdn_data = json.load(open(DATASET_INFO['CVDN']['path']))
    cvdn_word_lens = []
    for item in cvdn_data:
        cvdn_word_lens.append(len(reconstruct_cvdn_instruction(item).split()))
    rows.append({
        'dataset': 'CVDN',
        'instances_after_loading': len(cvdn_word_lens),
        'len_unit': 'whitespace words in reconstructed text',
        'min': min(cvdn_word_lens),
        'median': statistics.median(cvdn_word_lens),
        'max': max(cvdn_word_lens),
        'truncation_rule': '__getitem__ 里 instruction 文本截到 128 词',
        'truncated_count': sum(x > 128 for x in cvdn_word_lens),
        'note': '但 instr_encoding 在 load_data 时生成，__getitem__ 不会同步重算',
    })

    soon_lens = []
    soon_item_count = 0
    with jsonlines.open(DATASET_INFO['SOON']['path']) as reader:
        for item in reader:
            soon_item_count += 1
            soon_lens.extend(len(enc['full']) for enc in item['instr_encodings'])
    rows.append({
        'dataset': 'SOON',
        'instances_after_loading': len(soon_lens),
        'len_unit': 'tokenized ids',
        'min': min(soon_lens),
        'median': statistics.median(soon_lens),
        'max': max(soon_lens),
        'truncation_rule': "item['instr_encodings'][j]['full'][:100]",
        'truncated_count': sum(x > 100 for x in soon_lens),
        'note': f'原始 item 数 {soon_item_count}，拆成指令样本后 {len(soon_lens)}',
    })

    return pd.DataFrame(rows)


stats_df = collect_stats()
stats_df

In [ ]:
goal_df = pd.DataFrame([
    {
        'dataset': 'R2R',
        'what agent must do': '沿自然语言路线走到终点附近',
        'success in SAME eval': 'nav_error < 3m，oracle_success 看轨迹上最近点',
        'object prediction needed': '否',
    },
    {
        'dataset': 'REVERIE',
        'what agent must do': '走到某个能看见目标物体的 viewpoint',
        'success in SAME eval': '最终 viewpoint 落在 goal_viewpoints；另算 rgs / rgspl',
        'object prediction needed': '是，需要 pred_objid',
    },
    {
        'dataset': 'CVDN',
        'what agent must do': '根据对话找到目标房间',
        'success in SAME eval': 'nav_error < 3m；还看 oracle path success 和 dist_to_end_reduction',
        'object prediction needed': '否',
    },
    {
        'dataset': 'SOON',
        'what agent must do': '先导航到目标 viewpoint，再指出目标方向/位置',
        'success in SAME eval': '导航 success + det_success，后者要求预测点落在目标 polygon',
        'object prediction needed': '是，需要 pred_obj_direction=(heading, elevation)',
    },
])

goal_df

## 结论速记

- `R2R`：最标准的单轮导航，目标是 viewpoint。
- `REVERIE`：语言里常包含目标物体，评估不仅看是否走到对的位置，还看物体识别结果 `pred_objid`。
- `CVDN`：原始监督信号是多轮对话，SAME 通过 `Question:` / `Answer:` 前缀把 `navigator` 和 `oracle` 两个说话人串成单条文本。
- `SOON`：本质上是“导航 + 指物”，既要到位，也要给出目标方向，评估里会检查预测点是否落在目标框多边形内。
- 一个值得单独留意的实现细节：`CVDN` 在 `__getitem__` 里把过长文本截成 128 个空格词，但没有同步重算 `instr_encoding`；而 `R2R` 虽然配置里写了 `max_instr_len: 80`，当前 `r2r.py` 的 `load_data` 并没有读取这个配置值，而是沿用函数默认参数 `max_instr_len=200`。

## 新问题验证 1：R2R / REVERIE / SOON 的多 instruction 到底怎么拆

这三类数据不是把一段长文本硬切成碎片，而是原始标注文件本来就有一个 `instructions` 列表。SAME 只是做逐条展开：

- 保留同一个 `path` / `scan` / `path_id` / `objId` 等上下文不变。
- 对 `instructions[j]` 逐条复制出一个新样本。
- 同时取与之对齐的 `instr_encodings[j]` 作为该条 instruction 的编码。
- 新样本的 `instr_id` 只是在原样本 ID 后加上 `j`，例如 `r2r_{path_id}_{j}`、`reverie_{path_id}_{objId}_{j}`、`soon_{i}_{path_id}_{j}`。

因此，单条 instruction 是否完整、是否可执行，取决于原始数据集本身的标注，而不是 SAME 的拆分逻辑。

In [ ]:
r2r_full = json.load(open(DATASET_INFO['R2R']['path']))
reverie_full = json.load(open(DATASET_INFO['REVERIE']['path']))
soon_items = list(jsonlines.open(DATASET_INFO['SOON']['path']).iter())

def inspect_instruction_splitting():
    rows = []
    for idx in [0, 1, 2, 10]:
        item = r2r_full[idx]
        for j, instr in enumerate(item['instructions']):
            rows.append({
                'dataset': 'R2R',
                'item_index': idx,
                'group_id': item['path_id'],
                'instruction_index': j,
                'instruction': instr,
                'encoding_len': len(item['instr_encodings'][j]),
            })
    for idx in [0, 1, 2, 10]:
        item = reverie_full[idx]
        group_id = f"{item.get('path_id', item.get('id'))} / obj={item.get('objId')}"
        for j, instr in enumerate(item['instructions']):
            rows.append({
                'dataset': 'REVERIE',
                'item_index': idx,
                'group_id': group_id,
                'instruction_index': j,
                'instruction': instr,
                'encoding_len': len(item['instr_encodings'][j]),
            })
    for idx in [0, 1, 2, 10]:
        item = soon_items[idx]
        for j, instr in enumerate(item['instructions']):
            rows.append({
                'dataset': 'SOON',
                'item_index': idx,
                'group_id': item['path_id'],
                'instruction_index': j,
                'instruction': instr['full'],
                'encoding_len': len(item['instr_encodings'][j]['full']),
            })
    return pd.DataFrame(rows)

split_df = inspect_instruction_splitting()
pd.set_option('display.max_colwidth', 160)
split_df

从上表可以直接看到：

- `R2R` 和 `REVERIE` 的每个原始 item 通常包含 2 到 3 条互相独立的完整指令改写，它们共享同一条路径或同一目标物体。
- `SOON` 的 `instructions` 通常只有 1 条，少数 item 有 2 条；每条都是完整句子，不是长文本碎片。
- 所以 `Hello` 这种“拆出来就失去上下文”的问题，不属于这三类数据的拆分机制。

## 新问题验证 2：CVDN 的 `The goal room contains a ...` 是怎么来的

这句话不是原始 `dialog_history` 中的某一轮，而是 `cvdn.py` 重建 instruction 时手工加上的模板前缀：

- 若 `dialog_history` 为空，instruction 只有这句。
- 若 `dialog_history` 不为空，也会先加这句，再把每轮对话改写成 `Question:` / `Answer:`。
- 其中 `a {target}` 是模板硬拼，所以在 `fireplace / picture / towel` 这类目标上会显得有些机械。

下面统计它是不是常见情况，并多看几个样本。

In [ ]:
cvdn_data = json.load(open(DATASET_INFO['CVDN']['path']))

def first_oracle_message(item):
    for turn in item['dialog_history']:
        if turn['role'] == 'oracle':
            return turn['message'].strip().lower()
    return None

cvdn_summary = {
    'num_samples': len(cvdn_data),
    'num_samples_with_target': sum(1 for item in cvdn_data if item.get('target')),
    'num_empty_dialog': sum(1 for item in cvdn_data if len(item['dialog_history']) == 0),
    'num_first_oracle_is_hello_like': sum(
        1 for item in cvdn_data if first_oracle_message(item) in {'hello', 'hi', 'hey', 'hello.', 'hi.'}
    ),
}
cvdn_summary

In [ ]:
example_ids = [0, 1, 2, 3, 4, 5, 20, 50]
rows = []
for idx in example_ids:
    item = cvdn_data[idx]
    rows.append({
        'inst_idx': item['inst_idx'],
        'target': item['target'],
        'dialog_turns': len(item['dialog_history']),
        'reconstructed_instruction': reconstruct_cvdn_instruction(item),
    })
pd.DataFrame(rows)

这说明：

- `The goal room contains a X.` 是 CVDN 全量样本里的通用前缀，不是个例。
- 它不太通顺，主要是因为模板固定而 `target` 常常只是单个名词。
- 对话本身也确实可能带有寒暄、拼写错误、短回答等噪声；SAME 不会进一步清洗。

## 新问题验证 3：SOON 的 `instruction_keys` / `instruction_full` 是否来自原始数据

是的，这几个字段就是 SOON 标注文件本身带的。SAME 在 `soon.py` 中直接读取：

- `instruction = item['instructions'][j]['full']`
- `instr_encoding = item['instr_encodings'][j]['full'][:100]`

所以 `attr / relation / region / nb_region / full` 都是 SOON annotation schema 的一部分。

In [ ]:
soon_key_rows = []
for idx in [0, 1, 2, 10]:
    item = soon_items[idx]
    soon_key_rows.append({
        'item_index': idx,
        'path_id': item['path_id'],
        'num_instructions': len(item['instructions']),
        'instruction_keys': list(item['instructions'][0].keys()),
        'instruction_full': item['instructions'][0]['full'],
    })
pd.DataFrame(soon_key_rows)

In [ ]:
from collections import Counter

num_instr_counter = Counter(len(item['instructions']) for item in soon_items)
keyset_counter = Counter(tuple(sorted(instr.keys())) for item in soon_items for instr in item['instructions'])

display(pd.DataFrame({'num_instructions': list(num_instr_counter.keys()), 'count': list(num_instr_counter.values())}).sort_values('num_instructions'))
display(pd.DataFrame({'instruction_keyset': [list(k) for k in keyset_counter.keys()], 'count': list(keyset_counter.values())}))

结论是：至少在当前 `val_unseen_house_enc_pseudo_obj_ade30k_label.jsonl` 里，所有 instruction 都有同一组键，而且绝大多数 item 只有 1 条 instruction。

## 新问题验证 4：`run_same.py --config configs/same/val_r2r_eval_only.yaml` 真实会对谁截断

这里要分两层：

1. 这条命令真实会加载哪些数据集。
2. SAME 代码对四类数据集分别写了什么截断逻辑。

先看第 1 层。

In [ ]:
from omegaconf import OmegaConf

default_cfg = OmegaConf.load(ROOT / 'third_party' / 'SAME' / 'src' / 'configs' / 'default.yaml')
r2r_eval_cfg = OmegaConf.load(ROOT / 'configs' / 'same' / 'val_r2r_eval_only.yaml')
merged_cfg = OmegaConf.merge(default_cfg, r2r_eval_cfg)

pd.DataFrame([
    {'field': 'task.source', 'value': list(merged_cfg.task.source)},
    {'field': 'task.val_source', 'value': list(merged_cfg.task.val_source)},
    {'field': 'task.test_source', 'value': list(merged_cfg.task.test_source)},
    {'field': 'model.max_instr_len', 'value': merged_cfg.model.max_instr_len},
])

这说明 `python scripts/experiments/run_same.py --config configs/same/val_r2r_eval_only.yaml` 实际上只会跑 `R2R`。因此，这条命令真正会遇到的截断只和 `R2R` 有关。不过为了完整起见，下面仍把四类数据都放在一张表里。

In [ ]:
trunc_rows = []

r2r_raw_lens = [len(enc) for item in r2r_full for enc in item['instr_encodings']]
trunc_rows.append({
    'dataset': 'R2R',
    'loaded_by_val_r2r_eval_only': True,
    'effective_cutoff_in_current_code': 200,
    'where_applied': 'r2r.py::load_data',
    'actual_truncated_count_on_val_unseen': sum(x > 200 for x in r2r_raw_lens),
    'note': f"config 中 model.max_instr_len=80，但当前 loader 未使用；val_unseen 中 >80 的有 {sum(x > 80 for x in r2r_raw_lens)} 条",
})

reverie_raw_lens = [len(enc) for item in reverie_full for enc in item['instr_encodings']]
trunc_rows.append({
    'dataset': 'REVERIE',
    'loaded_by_val_r2r_eval_only': False,
    'effective_cutoff_in_current_code': None,
    'where_applied': 'reverie.py::load_data',
    'actual_truncated_count_on_val_unseen': 0,
    'note': f'max raw length={max(reverie_raw_lens)}',
})

cvdn_word_lens = [len(reconstruct_cvdn_instruction(item).split()) for item in cvdn_data]
trunc_rows.append({
    'dataset': 'CVDN',
    'loaded_by_val_r2r_eval_only': False,
    'effective_cutoff_in_current_code': 128,
    'where_applied': 'cvdn.py::__getitem__',
    'actual_truncated_count_on_val_unseen': sum(x > 128 for x in cvdn_word_lens),
    'note': '按空格词截 instruction 文本，但不会同步重算 instr_encoding',
})

soon_raw_lens = [len(enc['full']) for item in soon_items for enc in item['instr_encodings']]
trunc_rows.append({
    'dataset': 'SOON',
    'loaded_by_val_r2r_eval_only': False,
    'effective_cutoff_in_current_code': 100,
    'where_applied': 'soon.py::load_data',
    'actual_truncated_count_on_val_unseen': sum(x > 100 for x in soon_raw_lens),
    'note': f'max raw length={max(soon_raw_lens)}',
})

pd.DataFrame(trunc_rows)